# 09 — DistilBERT Routing Arm (all-in-notebook)

Self-contained fine-tuning of DistilBERT on the nested JSONL from notebook 07, with
**class-weighted** loss (IRS is rare) and 512-token truncation. Best run on a **GPU
(Colab)**. The cell below auto-detects the environment: it trains when `torch` +
`transformers` are available, otherwise it prints exact run instructions instead of
failing.


## 1. Load splits + environment check

In [1]:
try:
    import google.colab  # noqa
    !pip -q install torch transformers datasets accelerate scikit-learn pandas
except Exception:
    pass

import json
from pathlib import Path
import numpy as np
import pandas as pd

def find_ml_dir():
    candidates = [Path("/content/drive/MyDrive/newstart_ai"), Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        if (root / "data" / "ml" / "train.jsonl").exists():
            return root / "data" / "ml"
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/newstart_ai/data/ml")
    except Exception:
        raise FileNotFoundError("Run notebook 07 first to create data/ml/*.jsonl")

ML = find_ml_dir()
REPORTS = ML.parent.parent / "reports"; REPORTS.mkdir(parents=True, exist_ok=True)
MODELS = ML.parent.parent / "models"; MODELS.mkdir(parents=True, exist_ok=True)

label_map = json.loads((ML / "label_map.json").read_text())
id2label = {v: k for k, v in label_map.items()}
labels_sorted = [id2label[i] for i in range(len(label_map))]
class_weights = json.loads((ML / "class_weights.json").read_text())["by_id"]

# Toggle: include the nested instruction text as extra context for the classifier.
USE_INSTRUCTIONS = False

def load_split(name):
    df = pd.read_json(ML / f"{name}.jsonl", lines=True)
    def build_text(row):
        t = str(row["text"])
        if USE_INSTRUCTIONS and isinstance(row.get("associated_instructions"), list):
            t += "\n" + "\n".join(a.get("text", "") for a in row["associated_instructions"])
        return t
    df["model_text"] = df.apply(build_text, axis=1)
    return df

train, val, test = load_split("train"), load_split("val"), load_split("test")
print("loaded:", len(train), len(val), len(test), "| labels:", labels_sorted)
print("USE_INSTRUCTIONS =", USE_INSTRUCTIONS)

import importlib.util
HAS_TORCH = importlib.util.find_spec("torch") is not None
HAS_TF = importlib.util.find_spec("transformers") is not None
print("torch:", HAS_TORCH, "| transformers:", HAS_TF)

loaded: 416 89 82 | labels: ['DMV', 'IRS', 'SSA', 'USCIS']
USE_INSTRUCTIONS = False
torch: False | transformers: False


## 2. Shared evaluation helper

In [2]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def evaluate(y_true, y_pred, arm, proba=None, threshold=0.0):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    ids = list(range(len(labels_sorted)))
    mask = np.ones(len(y_true), bool)
    if proba is not None and threshold > 0:
        mask = proba.max(axis=1) >= threshold
    yt, yp = y_true[mask], y_pred[mask]
    rep = classification_report(yt, yp, labels=ids, target_names=labels_sorted, output_dict=True, zero_division=0)
    m = {
        "arm": arm, "n_test": int(len(y_true)), "n_scored": int(mask.sum()),
        "coverage": round(float(mask.mean()), 4),
        "accuracy": round(float(accuracy_score(yt, yp)), 4),
        "macro_f1": round(float(f1_score(yt, yp, labels=ids, average="macro", zero_division=0)), 4),
        "per_class_f1": {l: round(rep[l]["f1-score"], 4) for l in labels_sorted},
        "per_class_recall": {l: round(rep[l]["recall"], 4) for l in labels_sorted},
        "confusion_matrix": confusion_matrix(yt, yp, labels=ids).tolist(),
        "labels": labels_sorted,
    }
    return m

def print_confusion(m):
    w = max(len(l) for l in labels_sorted) + 1
    print(" " * (w + 1) + " ".join(f"{l[:6]:>6}" for l in labels_sorted))
    for i, row in enumerate(m["confusion_matrix"]):
        print(f"{labels_sorted[i]:<{w}} " + " ".join(f"{v:>6}" for v in row))

## 3. Fine-tune (runs when torch + transformers are present)

In [3]:
MODEL_NAME, MAX_LEN, EPOCHS, BATCH, LR = "distilbert-base-uncased", 512, 4, 8, 5e-5

if HAS_TORCH and HAS_TF:
    import torch
    from torch import nn
    from datasets import Dataset
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              DataCollatorWithPadding, Trainer, TrainingArguments)
    from sklearn.metrics import f1_score

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    def to_ds(df):
        enc = tok(list(df["model_text"]), truncation=True, max_length=MAX_LEN)
        enc["labels"] = list(df["label"]); return Dataset.from_dict(enc)
    ds_tr, ds_va, ds_te = to_ds(train), to_ds(val), to_ds(test)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels_sorted),
        id2label={i: l for i, l in enumerate(labels_sorted)}, label2id=label_map)

    w = torch.tensor([float(class_weights[str(i)]) for i in range(len(labels_sorted))], dtype=torch.float)
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop("labels"); out = model(**inputs)
            loss = nn.CrossEntropyLoss(weight=w.to(out.logits.device))(out.logits, labels)
            return (loss, out) if return_outputs else loss
    def compute_metrics(p):
        return {"macro_f1": f1_score(p.label_ids, np.argmax(p.predictions, 1), average="macro", zero_division=0)}

    args = TrainingArguments(output_dir=str(MODELS / "distilbert" / "ckpt"),
        num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        learning_rate=LR, weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1", logging_steps=20, report_to=[], seed=42)
    trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tok))
    trainer.train()

    def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
    test_proba = softmax(trainer.predict(ds_te).predictions)
    y_pred = test_proba.argmax(1)
    metrics = evaluate(test["label"].values, y_pred, arm="distilbert")
    print("macro-F1:", metrics["macro_f1"], "| accuracy:", metrics["accuracy"])
    print_confusion(metrics)
    trainer.save_model(str(MODELS / "distilbert" / "model")); tok.save_pretrained(str(MODELS / "distilbert" / "model"))
    (REPORTS / "metrics_distilbert.json").write_text(json.dumps(metrics, indent=2))
    print("saved ->", REPORTS / "metrics_distilbert.json")
else:
    print("Transformers stack not installed here. To fine-tune (Colab GPU recommended):")
    print("  pip install torch transformers datasets accelerate")
    print("  then Runtime > Run all. Config:", dict(model=MODEL_NAME, max_len=MAX_LEN, epochs=EPOCHS))

Transformers stack not installed here. To fine-tune (Colab GPU recommended):
  pip install torch transformers datasets accelerate
  then Runtime > Run all. Config: {'model': 'distilbert-base-uncased', 'max_len': 512, 'epochs': 4}


## 4. Notes

- Imbalance handled by weighted cross-entropy (weights from `class_weights.json`).
- `MAX_LEN=512` keeps the discriminative form header; the nested instruction text is
  included only if you set `USE_INSTRUCTIONS = True` in cell 1.
- With IRS at ~14 forms, expect noisy IRS metrics — the number to watch is IRS recall.
